# Python's Not Slow

You are, and your code probably sucks. 

We operate on different time scales, and the ability to switch seamlessly  
is crucial as a civil engineer.

Consider a comparative timescale:

One nanosecond (a computer's "tick" or sense of time) is to one second (our  
"tick") as one second is to 31.7 years to us.

The 1-second function we're talking about optimizing? That's a billion  
operations for your processor. It's the computer-equivalent of you working  
on a single task for over three decades. When we're planning 10-year bridge  
mega-projects, obsessing over a few hundred milliseconds is often less than  
a rounding error, it's a downright waste of time.

Before continuing down this path or accepting "python is slow" as fact,  
let's consider what you're really doing. Is your application reliant on a  
"slow" or unresponsive system, like an API call that's taking a long time  
to respond? Can your process be improved by more efficiently querying this  
system? Is it your code itself? Have you traced it to see where the  
inefficiencies are? Are you actually just here because of internet graphics  
like this one, and you thought this notebook would reaffirm your  
pre-existing bias?

!["A graph showing the speed of various programming languages"](https://miro.medium.com/1*NkA6YBauJDXZs_qjVtauag.gif)

If that's the case, you're probably wasting your time or being pedantic.  
Rethink your choices. Here's a reality check: our job isn't to write "for  
loops as a service".

If you really insist on continuing down this nightmare of a rabbit hole,  
we're skipping those "medium level" languages like Java, Go and C# and  
dancing with the beast. Unless you have a reason to target a specific  
system like windows, android or MacOS, don't.

(If anyone wants to rewrite the guide in Rust, I'm open to it.)

# The C Programming Language

This is your last gentle reminder that this is hello world in python,

```python
print("Hello World")
```

This is C#,

```C#
using System;

class Program
{
    static void Main(string[] args)
    {
        Console.WriteLine("Hello World");
    }
}
```

C++;

```cpp
#include <iostream>

int main() {
    std::cout << "Hello World" << std.endl;
    return 0;
}
```

Rust;

```Rust
fn main() {
    println!("Hello World");
}
```

and C

```C
#include <stdio.h>

int main() {
    printf("Hello World\n");
    return 0;
}
```

The reason we recommend python for development in most situations is  
it's much easier to read and debug, which reduces a lot of reliance on  
advanced development tools and need for advanced computer science knowledge  
to be "proficient" in the language.

# Wrapping C Functions for Python

Fine. Here's the problem outline, we want an algorithm to find "k" number  
of "nearest neighbors" of a provided point in 2D space given a large  
dataset of points.

The real world representation of this would be for GIS data. Consider a  
bridge and you want to find either nearby bridges or other geographical  
features located closely to the structure.

Here is a basic outline of the python implementation of something like this  
along with some randomly generated dummy data;

In [1]:
import numpy as np
import math
import time

# Generate a large number of points to search within, 500k gets pretty slow
num_search_points = 60000
search_points = np.random.rand(num_search_points, 2)

# Generate a smaller set of points to find neighbors for
num_query_points = 100
query_points = np.random.rand(num_query_points, 2)

k = 5

In [2]:
def euclidean_distance_py(p1, p2):
    """Calculates Euclidean distance between two points in pure Python."""
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def knn_search_py(query_points, search_points, k):
    """
    Finds the K-Nearest Neighbors for each query point from a set of search points.
    This is a naive, pure-Python implementation to demonstrate slowness.
    """
    all_neighbors = []
    for query_point in query_points:
        distances = []
        for search_point in search_points:
            dist = euclidean_distance_py(query_point, search_point)
            distances.append((dist, search_point))

        # Sort all distances to find the nearest
        distances.sort(key=lambda x: x[0])

        # Extract the top k neighbors
        neighbors = [point for dist, point in distances[:k]]
        all_neighbors.append(neighbors)

    return all_neighbors

In [3]:
%%time

# Run the slow KNN search.
# The '%%time' magic command will measure the execution time of this cell.
# A slow result demonstrates the performance of this pure Python implementation.
# Wall time represents the amount of time that passes on a clock on the wall
# CPU is the time your CPU experiences (Since multiple cores can work together)
nearest_neighbors = knn_search_py(query_points, search_points, k)

print(f"Found neighbors for {len(nearest_neighbors)} query points.")

Found neighbors for 100 query points.
CPU times: total: 6.55 s
Wall time: 6.59 s


So as mentioned, we just made that computer suffer for ~7.52s or it's  
equivalent of 238 human years... Let's bring it down utilizing C,

(interestingly this notebook also serves as benchmark of your CPUs  
performance)

```C/C++
// File: knn.c

#include <math.h>
#include <stdlib.h>
#include <stdio.h>

/*
 * This C code is designed to be compiled into a shared library (.so or .dll)
 * and called from Python using the ctypes library. It provides a significant
 * speed-up for a K-Nearest Neighbors (KNN) search by leveraging C's performance
 * for looping and calculations.
 */

// Helper structure to keep track of a point's distance from the query point
// and its original index in the search array.
typedef struct {
    double distance;
    int index;
} DistanceIndex;

// A comparison function required by C's standard `qsort` function.
// It tells qsort how to compare two DistanceIndex structs so they can be
// sorted by their `distance` member in ascending order.
int compare_distance_index(const void *a, const void *b) {
    DistanceIndex *di_a = (DistanceIndex *)a;
    DistanceIndex *di_b = (DistanceIndex *)b;
    if (di_a->distance < di_b->distance) return -1;
    if (di_a->distance > di_b->distance) return 1;
    return 0;
}

// A helper function to calculate the squared Euclidean distance between two 2D points.
// Calculating the square root is computationally expensive, so we can often avoid it.
// By comparing squared distances, we get the same ordering of nearest neighbors
// without needing to perform a `sqrt` for every distance calculation.
double euclidean_distance_sq(double p1_x, double p1_y, double p2_x, double p2_y) {
    double dx = p1_x - p2_x;
    double dy = p1_y - p2_y;
    return dx * dx + dy * dy;
}


// The main KNN search function exposed to Python.
// It processes all query points in a single call to minimize Python-C overhead.
//
// Parameters:
//   - query_points: A pointer to a flat array of doubles representing the query points (e.g., [x1, y1, x2, y2, ...]).
//   - num_query_points: The number of points in the `query_points` array.
//   - search_points: A pointer to a flat array of doubles representing the points to search within.
//   - num_search_points: The number of points in the `search_points` array.
//   - k: The number of nearest neighbors to find for each query point.
//   - results: An output pointer to a flat integer array. The function will fill this array
//              with the *indices* of the k-nearest neighbors for each query point.
//              The size of this array must be `num_query_points * k`.
void knn_search(double* query_points, int num_query_points,
                double* search_points, int num_search_points,
                int k, int* results) {

    // This check is important. If k is larger than the number of points to search,
    // it's an impossible request.
    if (k > num_search_points) {
        fprintf(stderr, "Error: k cannot be greater than the number of search points.\\n");
        return;
    }

    // Allocate memory to store the distances for a *single* query point against all search points.
    // We can reuse this memory for each query point.
    DistanceIndex *distances = malloc(num_search_points * sizeof(DistanceIndex));
    if (distances == NULL) {
        fprintf(stderr, "Error: Failed to allocate memory for distances.\\n");
        return;
    }

    // --- Main Loop: Iterate over each query point ---
    for (int i = 0; i < num_query_points; i++) {
        // Each point has 2 coordinates (x, y), so we access them by `i * 2` and `i * 2 + 1`.
        double qp_x = query_points[i * 2];
        double qp_y = query_points[i * 2 + 1];

        // --- Inner Loop: Calculate distance to each search point ---
        for (int j = 0; j < num_search_points; j++) {
            double sp_x = search_points[j * 2];
            double sp_y = search_points[j * 2 + 1];

            distances[j].distance = euclidean_distance_sq(qp_x, qp_y, sp_x, sp_y);
            distances[j].index = j; // Store the original index
        }

        // Sort the `distances` array based on the distance. After this, the first
        // `k` elements will be the nearest neighbors.
        qsort(distances, num_search_points, sizeof(DistanceIndex), compare_distance_index);

        // --- Store Results: Extract the top k indices ---
        for (int ki = 0; ki < k; ki++) {
            // The `results` array is a flat 1D array. We calculate the correct
            // position to store the index for the i-th query point's k-th neighbor.
            results[i * k + ki] = distances[ki].index;
        }
    }

    // Free the allocated memory to prevent memory leaks.
    free(distances);
}
```

After saving the above code as `knn.c`, it can be compiled utilizing the gcc compiler with the following code,

`gcc -shared -o knn.dll knn.c`

Running C code in python isn't as straightforward as importing a library,

In [4]:
import ctypes
import time

c_lib = ctypes.CDLL("./knn.dll")

c_lib.knn_search.argtypes = [
    ctypes.POINTER(ctypes.c_double), ctypes.c_int,
    ctypes.POINTER(ctypes.c_double), ctypes.c_int,
    ctypes.c_int, ctypes.POINTER(ctypes.c_int)
]
c_lib.knn_search.restype = None

In [5]:
# Create the C-compatible results buffer using only ctypes ---

# First, define the C array type: an array of C integers.
# The size is determined by the existing notebook variables.
ResultArrayType = ctypes.c_int * (num_query_points * k)

# Now, instantiate that type to allocate the actual contiguous block of memory.
c_results = ResultArrayType()

In [6]:
# --- Get pointers to the input data ---
# We still need to get pointers from the existing NumPy arrays for the input.
query_ptr = query_points.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
search_ptr = search_points.ctypes.data_as(ctypes.POINTER(ctypes.c_double))

In [7]:
# --- Call the C function and time it ---
print("Running KNN search using ctypes-allocated memory for results...")
start_time = time.time()

# Execute your C function. Note that `c_results` can be passed directly
# because it is already a ctypes object (a pointer to the array's start).
c_lib.knn_search(
    query_ptr,
    num_query_points,
    search_ptr,
    num_search_points,
    k,
    c_results
)

end_time = time.time()
print(f"Wall time: {end_time - start_time:.4f} s")

# The results are now populated in the `c_results` object.
# You can access them like a Python list if needed.
# print("\nFirst 5 result indices:", c_results[:5])

Running KNN search using ctypes-allocated memory for results...
Wall time: 0.7665 s


Don't get me wrong, I understand this 6.8 seconds or whatever can be a huge  
deal in an application with thousands or millions of loops. I'm just  
assuring you that in the absolutely unlikely event that python's speed is  
the ACTUAL problem, we have a solution, which we've just shown.

In reality if you google or ask an AI about this question from the start,  
it'll probably just direct you to utilize scikit learn, which is probably  
the "best" solution we'll be able to develop internally. Again, the actual  
issue is an algorithmic one.

In [8]:
%%time

from sklearn.neighbors import NearestNeighbors

# Create and "fit" the k-NN model using the existing notebook data.
# n_jobs=-1 tells scikit-learn to use all available CPU cores for the search.
nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', n_jobs=-1)
nbrs.fit(search_points)

# Run the search on the existing `query_points` data.
distances, indices = nbrs.kneighbors(query_points)

# Print a consistent output message for comparison with the other tests.
print(f"Found neighbors for {len(indices)} query points using scikit-learn.")

Found neighbors for 100 query points using scikit-learn.
CPU times: total: 781 ms
Wall time: 823 ms


Note that the above is down to milliseconds, almost 20x faster than even  
our C solution. There are situations where we may have to rewrite certain  
tools, but the number of edge cases between "rewrite" and "our approach is  
impractical/infeasible in any language" is insanely small.  I've  
legitimately never encountered a need for it in 8 years.